# Chain-of-Thought vs Extended Thinking
### A 2-step prompt chain, measured in tokens

**The scenario:** an agentic-style task where Call 2 must build on Call 1's reasoning.

> A logistics planner is deciding whether a shipment qualifies for express handling.
> Call 1 analyzes the shipment (weight, distance, deadline) and reasons through the decision.
> Call 2 takes that reasoning and produces the final structured recommendation.

We run this **same problem statement** through two branches:

| Branch | Reasoning mechanism | How Call 2 gets Call 1's reasoning |
|---|---|---|
| **A. Plain CoT** | `"Think step by step"` in the prompt, no `thinking` param | You manually copy the **entire text response** back into Call 2's message history |
| **B. Extended Thinking** | `thinking: {type: "adaptive"}` | You pass the **structured `thinking` block(s)** back verbatim; the API manages/prunes what's billed |

### What this notebook will (and won't) prove

- ✅ It will show the **mechanical difference**: CoT reasoning is plain text you own and must re-paste; extended-thinking reasoning is an opaque block the API validates and manages for you.
- ✅ It will show real `usage` token counts for both branches, side by side.
- ✅ It will show the API **rejecting** a tampered thinking block — proof these blocks are integrity-checked, not just cosmetic.
- ⚠️ It will **not** show `redacted_thinking` as a "lighter/compressed" version of reasoning. Redaction is an *encrypted* safety fallback that appears occasionally when Claude's internal reasoning gets flagged — it's the same reasoning, encrypted, not a smaller one. You can't reliably force it (Anthropic ships a magic test string for that, used below only as a demo).
- ⚠️ Over just 2 calls, raw token savings from extended thinking may be modest — the API's automatic pruning of old thinking blocks is a benefit that compounds over **longer** multi-turn conversations, not necessarily this single round trip. The notebook prints the numbers so you can see this honestly rather than assuming a dramatic gap.

## 0. Setup

In [2]:
%pip install -q anthropic matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
import copy
from anthropic import Anthropic

# Set your key as an env var before launching VS Code / Jupyter:
#   export ANTHROPIC_API_KEY=sk-ant-...
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

MODEL = "claude-sonnet-4-6"
MAX_TOKENS = 2048

PROBLEM_STATEMENT = (
    "A shipment weighs 42kg, needs to travel 1,850km, and the customer's "
    "deadline is 36 hours from now. Our express carrier can guarantee "
    "delivery in 30 hours for shipments under 50kg travelling under 2,000km, "
    "but express slots cost 3x standard shipping. Standard shipping takes "
    "48 hours regardless of weight or distance under 2,500km. "
    "Decide whether this shipment needs express handling, and justify it."
)

print("Problem statement loaded. Model:", MODEL)

Problem statement loaded. Model: claude-sonnet-4-6


## Branch A — Plain Chain-of-Thought

No `thinking` param. We just ask Claude to reason step by step in its normal text output.
The **entire text** (reasoning + answer) is regular output content — there's no separation
between "scratchpad" and "final answer" as far as the API is concerned.

In [4]:
cot_call_1 = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    messages=[
        {
            "role": "user",
            "content": (
                "Think step by step and show your full reasoning before you "
                "answer.\n\n" + PROBLEM_STATEMENT
            ),
        }
    ],
)

cot_call_1_text = "".join(block.text for block in cot_call_1.content if block.type == "text")

print("--- Call 1 response (truncated) ---")
print(cot_call_1_text[:600], "...\n")
print("Call 1 usage:", cot_call_1.usage)

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
# Call 2: we MUST manually paste the entire Call 1 text back in to preserve
# continuity. This is the "increased token cost" mechanism you described —
# it's just normal message history, but that history contains the full
# reasoning transcript, verbatim, every time.
cot_messages = [
    {"role": "user", "content": PROBLEM_STATEMENT},
    {"role": "assistant", "content": cot_call_1_text},
    {
        "role": "user",
        "content": (
            "Based on your analysis above, give me ONLY the final decision "
            "in this exact format: 'Decision: <EXPRESS|STANDARD>. Reason: <one sentence>.'"
        ),
    },
]

cot_call_2 = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=cot_messages,
)

cot_call_2_text = "".join(block.text for block in cot_call_2.content if block.type == "text")
print("--- Call 2 final answer ---")
print(cot_call_2_text)
print("\nCall 2 usage:", cot_call_2.usage)

## Branch B — Extended Thinking (adaptive)

Same problem statement. `thinking.type = "adaptive"` lets Claude decide how much to think.
The reasoning comes back as a **structured `thinking` block** (with an opaque `signature`),
separate from the `text` block that holds the visible answer.

In [ ]:
ext_call_1 = client.messages.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    thinking={"type": "adaptive"},
    output_config={"effort": "high"},
    messages=[{"role": "user", "content": PROBLEM_STATEMENT}],
)

thinking_blocks = [b for b in ext_call_1.content if b.type in ("thinking", "redacted_thinking")]
text_blocks = [b for b in ext_call_1.content if b.type == "text"]

print(f"Content blocks returned: {[b.type for b in ext_call_1.content]}\n")
if thinking_blocks and thinking_blocks[0].type == "thinking":
    print("--- Thinking block (truncated) ---")
    print(thinking_blocks[0].thinking[:500], "...\n")
print("--- Visible answer ---")
print("".join(b.text for b in text_blocks)[:500])
print("\nCall 1 usage:", ext_call_1.usage)

In [ ]:
# Call 2: pass the ENTIRE assistant message back verbatim — thinking blocks
# included, unmodified, in the exact order received. This is a hard
# requirement: the API rejects reordered/edited/partially-dropped thinking
# blocks with a 400 error (demonstrated further down).
#
# We don't hand-copy any text here — we just echo the SDK's own message
# object back. The API internally decides which thinking blocks it still
# needs and bills input tokens only for those it actually shows to Claude.
ext_messages = [
    {"role": "user", "content": PROBLEM_STATEMENT},
    {"role": "assistant", "content": ext_call_1.content},
    {
        "role": "user",
        "content": (
            "Based on your analysis above, give me ONLY the final decision "
            "in this exact format: 'Decision: <EXPRESS|STANDARD>. Reason: <one sentence>.'"
        ),
    },
]

ext_call_2 = client.messages.create(
    model=MODEL,
    max_tokens=300,
    thinking={"type": "adaptive"},
    output_config={"effort": "high"},
    messages=ext_messages,
)

ext_call_2_text = "".join(b.text for b in ext_call_2.content if b.type == "text")
print("--- Call 2 final answer ---")
print(ext_call_2_text)
print("\nCall 2 usage:", ext_call_2.usage)

## Side-by-side token comparison

In [ ]:
def usage_dict(u):
    return {
        "input_tokens": u.input_tokens,
        "output_tokens": u.output_tokens,
        "cache_read_input_tokens": getattr(u, "cache_read_input_tokens", 0) or 0,
    }

results = {
    "CoT - Call 1": usage_dict(cot_call_1.usage),
    "CoT - Call 2": usage_dict(cot_call_2.usage),
    "ExtThinking - Call 1": usage_dict(ext_call_1.usage),
    "ExtThinking - Call 2": usage_dict(ext_call_2.usage),
}

print(f"{'Call':<24}{'input_tokens':>14}{'output_tokens':>15}")
print("-" * 53)
for name, u in results.items():
    print(f"{name:<24}{u['input_tokens']:>14}{u['output_tokens']:>15}")

cot_total_input = results["CoT - Call 1"]["input_tokens"] + results["CoT - Call 2"]["input_tokens"]
ext_total_input = results["ExtThinking - Call 1"]["input_tokens"] + results["ExtThinking - Call 2"]["input_tokens"]

print("\nTotal input tokens across both calls:")
print(f"  Plain CoT:         {cot_total_input}")
print(f"  Extended Thinking: {ext_total_input}")
print(
    f"\nNote: this is a 2-call snapshot. The gap widens as the conversation "
    f"grows longer, since the API keeps pruning old thinking blocks from the "
    f"billed input on Claude's side, while a plain-text CoT transcript just "
    f"keeps growing turn over turn unless YOU manually trim it."
)

In [ ]:
import matplotlib.pyplot as plt

labels = ["Call 1", "Call 2", "Total"]
cot_vals = [
    results["CoT - Call 1"]["input_tokens"],
    results["CoT - Call 2"]["input_tokens"],
    cot_total_input,
]
ext_vals = [
    results["ExtThinking - Call 1"]["input_tokens"],
    results["ExtThinking - Call 2"]["input_tokens"],
    ext_total_input,
]

x = range(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4.5), facecolor="white")
ax.set_facecolor("white")
ax.bar([i - width / 2 for i in x], cot_vals, width, label="Plain CoT", color="#4C72B0")
ax.bar([i + width / 2 for i in x], ext_vals, width, label="Extended Thinking", color="#DD8452")

ax.set_xticks(list(x))
ax.set_xticklabels(labels, color="black")
ax.set_ylabel("Input tokens", color="black")
ax.set_title("Input tokens billed: Plain CoT vs Extended Thinking", color="black")
ax.tick_params(colors="black")
for spine in ax.spines.values():
    spine.set_color("black")
ax.legend()
plt.tight_layout()
plt.show()

## Bonus 1 — proving the thinking block is integrity-checked

Tamper with a single character inside the returned thinking block and resend.
The docs say this must fail with a `400 invalid_request_error`. Let's confirm it.

In [ ]:
import anthropic

tampered_content = copy.deepcopy(ext_call_1.content)
for block in tampered_content:
    if block.type == "thinking":
        block.thinking = block.thinking + " (tampered)"
        break

tampered_messages = [
    {"role": "user", "content": PROBLEM_STATEMENT},
    {"role": "assistant", "content": tampered_content},
    {"role": "user", "content": "Now give me the final decision."},
]

try:
    client.messages.create(
        model=MODEL,
        max_tokens=300,
        thinking={"type": "adaptive"},
        messages=tampered_messages,
    )
    print("No error raised — unexpected.")
except anthropic.BadRequestError as e:
    print("Got the expected 400 error:")
    print(e)

## Bonus 2 — what `redacted_thinking` actually looks like

This uses Anthropic's documented magic test string to *force* a redacted block
on demand — in normal use, redaction is triggered occasionally and automatically
by safety systems, not something you control. This is for illustration only,
to show the block shape, not a technique to rely on in production.

In [ ]:
MAGIC_REDACT_STRING = (
    "ANTHROPIC_MAGIC_STRING_TRIGGER_REDACTED_THINKING_46C9A13E193C177646C7398A"
    "98432ECCCE4C1253D5E2D82641AC0E52CC2876CB"
)

redacted_demo = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    thinking={"type": "adaptive"},
    messages=[{"role": "user", "content": MAGIC_REDACT_STRING}],
)

for block in redacted_demo.content:
    print(block.type)

redacted = [b for b in redacted_demo.content if b.type == "redacted_thinking"]
if redacted:
    print("\nredacted_thinking content is opaque/encrypted, e.g.:")
    print(redacted[0].data[:80], "...")
    print(
        "\nThis is NOT a smaller/compressed version of the reasoning — it's the "
        "same reasoning, encrypted. You still pass it back verbatim, same as a "
        "normal thinking block, and Claude decrypts it internally to keep context."
    )
else:
    print("No redacted block this time — try again, behavior can vary by model version.")

## Takeaways for the article

1. **Mechanism, not magic**: Plain CoT reasoning is ordinary text you own and must manually re-paste into every subsequent call. Extended thinking reasoning comes back as a structured, signed block that you echo back verbatim — the API validates it and decides internally which parts still matter.
2. **`redacted_thinking` ≠ "lite" thinking.** It's an *encrypted*, occasional safety fallback — same reasoning, not a cheaper one. Don't build a cost argument around it.
3. **Where the real savings come from**: the API bills input tokens only for the thinking blocks it actually still needs across turns, and this is per-model and automatic. On a single 2-call round trip the gap may be small; it's a multi-turn, multi-tool-call agent loop where this compounds.
4. **Integrity, not convenience**: thinking blocks are tamper-evident (`signature` field). Edit one character and the whole request 400s. CoT text has no such protection — you could silently corrupt your own reasoning trail and the API would never know.
5. **Best real-world fit**: agentic tool-use loops, where the docs make preserving thinking blocks a hard requirement rather than a style choice — that's the strongest, most defensible example for your article.